In [2]:
import pandas as pd
import numpy as np
import warnings
import io

warnings.filterwarnings("ignore")

# ----------------------------------------
# Helper functions for log and summary writing
# ----------------------------------------
def write_md_header(f, title, level=1):
    f.write(f"{'#' * level} {title}\n\n")

def write_md_text(f, text):
    f.write(f"{text}\n\n")

def write_md_code_block(f, content):
    f.write(f"```\n{content}\n```\n\n")

def write_md_list(f, items):
    for item in items:
        f.write(f"- {item}\n")
    f.write("\n")

# ----------------------------------------
# 1. Data Acquisition
# ----------------------------------------
PATH = "vgchartz-2024.csv"
df = pd.read_csv(PATH)
raw_shape = df.shape

with open("preprocessing.md", "w", encoding="utf-8") as log_file:
    write_md_header(log_file, "Preprocessing Log")
    
    write_md_text(log_file, f"**Raw data loaded from:** `{PATH}`")
    write_md_text(log_file, f"**Initial shape:** {raw_shape[0]} rows, {raw_shape[1]} columns")
    write_md_text(log_file, "**Initial columns:** " + ", ".join(df.columns))

    # ----------------------------------------
    # 2. Column Selection
    # ----------------------------------------
    cols = [
        'console', 'genre', 'publisher', 'developer', 'critic_score',
        'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales'
    ]
    df = df[cols].copy()
    
    write_md_header(log_file, "Step 1: Column selection", level=2)
    write_md_text(log_file, "**Action:** Kept only analysis-relevant columns; removed `img` and other metadata.")
    write_md_text(log_file, "**Justification:** The `img` column contains image URLs and is not useful for statistical analysis. Other possible metadata columns were excluded for clarity.")
    write_md_text(log_file, f"**Shape after selection:** {df.shape}")

    # ----------------------------------------
    # 3. Missing Value Handling
    # ----------------------------------------
    before_dropna = df.shape[0]
    
    # Remove rows missing console or genre
    df = df.dropna(subset=['console', 'genre'])
    after_dropna_subset = df.shape[0]
    
    write_md_header(log_file, "Step 2: Row deletion based on missing values", level=2)
    write_md_text(log_file, f"**Action:** Removed rows where `console` or `genre` was missing.")
    write_md_text(log_file, f"**Rows before:** {before_dropna}")
    write_md_text(log_file, f"**Rows after:** {after_dropna_subset}")
    write_md_text(log_file, "**Justification:** `console` and `genre` are primary categorical grouping variables. They cannot be reliably imputed; removing these rows avoids introducing false categories.")
    
    # Impute critic_score with median
    critic_median = df['critic_score'].median()
    df['critic_score'] = df['critic_score'].fillna(critic_median)
    
    write_md_text(log_file, f"**Action:** Filled missing `critic_score` values with the median ({critic_median:.2f}).")
    write_md_text(log_file, "**Justification:** The critic score distribution is approximately symmetric, so the median is a robust imputation less affected by outliers.")

    # ----------------------------------------
    # 4. Business Rule Filtering
    # ----------------------------------------
    before_filter = df.shape[0]
    df = df.query('total_sales > 0')
    after_filter = df.shape[0]
    
    write_md_header(log_file, "Step 3: Sales filtering", level=2)
    write_md_text(log_file, "**Action:** Removed records with `total_sales <= 0`.")
    write_md_text(log_file, f"**Rows before:** {before_filter}")
    write_md_text(log_file, f"**Rows after:** {after_filter}")
    write_md_text(log_file, "**Justification:** Zero or negative sales values are likely data errors or placeholders; they have no analytical meaning and would distort summary statistics.")

    # ----------------------------------------
    # 5. Text Normalization
    # ----------------------------------------
    df['console'] = df['console'].str.strip().str.upper()
    df['genre'] = df['genre'].str.strip().str.upper()
    
    write_md_header(log_file, "Step 4: Text normalization", level=2)
    write_md_text(log_file, "**Action:** Converted `console` and `genre` to uppercase and removed leading/trailing whitespace.")
    write_md_text(log_file, "**Justification:** Prevents duplicate categories caused by inconsistent capitalisation or accidental spaces, ensuring clean grouping.")

    # ----------------------------------------
    # 6. Derived Variable
    # ----------------------------------------
    df['log_sales'] = np.log1p(df['total_sales'])
    
    write_md_header(log_file, "Step 5: Target variable transformation", level=2)
    write_md_text(log_file, "**Action:** Created `log_sales = log1p(total_sales)`.")
    write_md_text(log_file, "**Justification:** The logarithm reduces the skewness of the sales distribution, making it more suitable for linear models and visualizations. `log1p` is used for numerical stability (though no zero sales remain).")

    # Final shape
    write_md_text(log_file, f"**Final dataset shape:** {df.shape}")


# ----------------------------------------
# 7. Save Cleaned Dataset
# ----------------------------------------
cleaned_path = "cleaned_vgchartz.csv"
df.to_csv(cleaned_path, index=False)

# ----------------------------------------
# 8. Generate Data Quality Summary 
# ----------------------------------------
with open("summary.md", "w", encoding="utf-8") as summ_file:
    write_md_header(summ_file, "Data Quality Summary")
    
    # 8.1 Null rates
    write_md_header(summ_file, "1. Null Rates (Final Dataset)", level=2)
    null_rates = (df.isnull().sum() / len(df)) * 100
    null_table = "| Column | Null % |\n|--------|--------|\n"
    for col in df.columns:
        null_table += f"| {col} | {null_rates[col]:.2f}% |\n"
    write_md_text(summ_file, null_table)
    
    # 8.2 Class balance
    write_md_header(summ_file, "2. Class Balance", level=2)
    
    write_md_header(summ_file, "Console Distribution", level=3)
    console_counts = df['console'].value_counts().to_frame().to_markdown()
    write_md_text(summ_file, console_counts)
    
    write_md_header(summ_file, "Genre Distribution", level=3)
    genre_counts = df['genre'].value_counts().to_frame().to_markdown()
    write_md_text(summ_file, genre_counts)
    
    # 8.3 Outlier treatment
    write_md_header(summ_file, "3. Outlier Treatment", level=2)
    write_md_list(summ_file, [
        "**total_sales:** Values <= 0 removed; no capping/winsorizing applied. Log transformation reduces impact of extreme values.",
        "**critic_score:** Missing values imputed with median; score range confirmed to be within 0–100.",
        "**Regional sales (na_sales, jp_sales, pal_sales, other_sales):** Retained as-is. Further outlier investigation may be performed during EDA."
    ])
    
    # 8.4 Schema
    write_md_header(summ_file, "4. Data Schema", level=2)
    buffer = io.StringIO()
    df.info(buf=buffer)
    schema_info = buffer.getvalue()
    write_md_code_block(summ_file, schema_info)
    write_md_text(summ_file, "**Data types per column:**")
    dtypes_table = df.dtypes.to_frame().to_markdown()
    write_md_text(summ_file, dtypes_table)
